In [5]:
from pathlib import Path
import scanpy as sc
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

In [2]:
import sys
sys.path.append(str(Path("../src").resolve()))
from c9_snrnaseq.io_utils import PROJECT_ROOT, load_and_annotate_sheet, save_checkpoint, _stage_header, _log
from c9_snrnaseq.qc_low_quality_cell import (
    compute_qc_metrics,
    detect_low_quality_cells,
    remove_low_quality_cells,
    freeze_raw_counts,
)
from c9_snrnaseq.ambient_rna import (
    estimate_ambient_rna,
    remove_ambient_rna,
)
from c9_snrnaseq.doublets_removal import (
    detect_doublets,
    remove_doublets,
)

from c9_snrnaseq.expression_preprocessing import (
    normalize_counts,
    log_transform,
    select_hvgs,
    scale_expression
)

from c9_snrnaseq.dimensionality_reduction import (
    run_pca,
    build_neighbor_graph,
    run_umap
)

from c9_snrnaseq.annotation import (
    run_leiden_clustering,
    find_cluster_markers
)

from c9_snrnaseq.pipeline import (
    process_one_sample
)

In [3]:
adata_hm = sc.read_h5ad(PROJECT_ROOT / "data/processed/merged/adata_hm_annotated.h5ad")

print(adata_hm.shape)
print("Has cell_class_major_harmony:", "cell_class_major_harmony" in adata_hm.obs.columns)
print("Has cell_class_refined_harmony:", "cell_class_refined_harmony" in adata_hm.obs.columns)

(89819, 2000)
Has cell_class_major_harmony: True
Has cell_class_refined_harmony: True


In [4]:
sample_df = pd.read_csv(PROJECT_ROOT / "config/samples.csv")

cleaned_adatas = []

for _, row in sample_df.iterrows():
    sample_id = str(row["sample_id"])
    sample_dir = PROJECT_ROOT / "data/processed/per_sample" / sample_id
    matches = sorted(sample_dir.glob("*_after_doublet_removal.h5ad"))

    if not matches:
        raise FileNotFoundError(f"No after_doublet_removal checkpoint found for {sample_id}")
    if len(matches) > 1:
        raise FileExistsError(f"Multiple after_doublet_removal checkpoints found for {sample_id}: {matches}")

    ad = sc.read_h5ad(matches[0])

    # overwrite / enforce metadata from samples.csv
    ad.obs["sample"] = sample_id
    ad.obs["condition"] = row["condition"]
    ad.obs["dataset"] = row["dataset"]
    ad.obs["tissue"] = row["tissue"]
    ad.obs["source_name"] = row["source_name"]

    # preserve original barcode and create unique merge key
    ad.obs["barcode"] = ad.obs_names.astype(str)
    ad.obs["merge_key"] = ad.obs["sample"].astype(str) + "__" + ad.obs["barcode"].astype(str)
    ad.obs_names = ad.obs["merge_key"].values

    cleaned_adatas.append(ad)

adata_merged_pb = sc.concat(
    cleaned_adatas,
    join="outer",
    merge="same",
    fill_value=0
)

adata_merged_pb.obs_names_make_unique()

# standardise raw counts storage
if "raw_counts" in adata_merged_pb.layers:
    adata_merged_pb.layers["counts"] = adata_merged_pb.layers["raw_counts"].copy()
else:
    adata_merged_pb.layers["counts"] = adata_merged_pb.X.copy()

print(adata_merged_pb.shape)
print("X dtype:", adata_merged_pb.X.dtype)
print("counts layer exists:", "counts" in adata_merged_pb.layers)
print("samples:", sorted(adata_merged_pb.obs["sample"].astype(str).unique()))

(89819, 61552)
X dtype: float32
counts layer exists: True
samples: ['GSM5292143', 'GSM5292144', 'GSM5292145', 'GSM5292146', 'GSM5292147', 'GSM5292148', 'GSM5292156', 'GSM5292157', 'GSM5292174', 'GSM5292175', 'GSM5292176', 'GSM5292177']


In [6]:
adata_hm.obs["barcode"] = adata_hm.obs_names.astype(str)
adata_hm.obs["merge_key"] = adata_hm.obs["sample"].astype(str) + "__" + adata_hm.obs["barcode"].astype(str)

refined_map = adata_hm.obs.set_index("merge_key")["cell_class_refined_harmony"].to_dict()
major_map = adata_hm.obs.set_index("merge_key")["cell_class_major_harmony"].to_dict()

adata_merged_pb.obs["cell_class_refined_harmony"] = adata_merged_pb.obs["merge_key"].map(refined_map)
adata_merged_pb.obs["cell_class_major_harmony"] = adata_merged_pb.obs["merge_key"].map(major_map)

print(adata_merged_pb.obs["cell_class_major_harmony"].value_counts(dropna=False))

cell_class_major_harmony
Unknown                33403
Excitatory_neurons     19806
Microglia              17981
Astrocytes              6821
Inhibitory_neurons      5955
OPCs                    4284
NaN                      936
Endothelial              363
Vascular_mural_like      270
Name: count, dtype: int64


In [ ]:
print("Cells in pb object:", adata_merged_pb.n_obs)
print("Cells with transferred major label:", adata_merged_pb.obs["cell_class_major_harmony"].notna().sum())
print("Cells missing transferred major label:", adata_merged_pb.obs["cell_class_major_harmony"].isna().sum())

In [7]:
confident_classes = [
    "Excitatory_neurons",
    "Inhibitory_neurons",
    "Astrocytes",
    "Microglia",
    "OPCs",
    "Endothelial",
    "Vascular_mural_like",
]

adata_pb_conf = adata_merged_pb[
    adata_merged_pb.obs["cell_class_major_harmony"].isin(confident_classes)
].copy()

print(adata_pb_conf.obs["cell_class_major_harmony"].value_counts())

cell_class_major_harmony
Excitatory_neurons     19806
Microglia              17981
Astrocytes              6821
Inhibitory_neurons      5955
OPCs                    4284
Endothelial              363
Vascular_mural_like      270
Name: count, dtype: int64


In [8]:
def pseudobulk_by_sample(
    adata,
    group_value,
    groupby_col="cell_class_major_harmony",
    min_cells_per_sample=20,
    layer="counts",
):
    sub = adata[adata.obs[groupby_col] == group_value].copy()
    if sub.n_obs == 0:
        raise ValueError(f"No cells found for {groupby_col}={group_value!r}.")

    cell_counts = sub.obs["sample"].value_counts()
    keep_samples = cell_counts[cell_counts >= min_cells_per_sample].index.tolist()
    if not keep_samples:
        raise ValueError(
            f"No samples retained for {groupby_col}={group_value!r} "
            f"with min_cells_per_sample={min_cells_per_sample}."
        )

    sub = sub[sub.obs["sample"].isin(keep_samples)].copy()
    sample_order = sorted(sub.obs["sample"].astype(str).unique())

    pb_counts = []
    pb_meta = []

    for samp in sample_order:
        ss = sub[sub.obs["sample"].astype(str) == samp]
        matrix = ss.layers[layer] if layer in ss.layers else ss.X
        vec = np.asarray(matrix.sum(axis=0)).ravel()

        pb_counts.append(vec)
        pb_meta.append({
            "sample": samp,
            "condition": ss.obs["condition"].astype(str).iloc[0],
            "group": group_value,
            "groupby_col": groupby_col,
            "n_cells": ss.n_obs,
        })

    counts_df = pd.DataFrame(pb_counts, index=sample_order, columns=sub.var_names)
    meta_df = pd.DataFrame(pb_meta).set_index("sample")

    return counts_df, meta_df

In [10]:
RESULTS_DIR = PROJECT_ROOT / "results"
PSEUDOBULK_DIR = RESULTS_DIR / "pseudobulk"
PSEUDOBULK_DIR.mkdir(parents=True, exist_ok=True)

broad_main_classes = [
    "Excitatory_neurons",
    "Inhibitory_neurons",
    "Astrocytes",
    "Microglia",
    "OPCs",
]

for ct in broad_main_classes:
    counts_df, meta_df = pseudobulk_by_sample(
        adata_merged_pb,
        group_value=ct,
        groupby_col="cell_class_major_harmony",
        min_cells_per_sample=20,
        layer="counts",
    )

    counts_df.to_csv(PSEUDOBULK_DIR / f"{ct}_counts.csv")
    meta_df.to_csv(PSEUDOBULK_DIR / f"{ct}_meta.csv")

    print(f"\nSaved pseudobulk for {ct}")
    print(counts_df.shape)
    print(meta_df)



Saved pseudobulk for Excitatory_neurons
(12, 61552)
           condition               group               groupby_col  n_cells
sample                                                                     
GSM5292143      sALS  Excitatory_neurons  cell_class_major_harmony     2867
GSM5292144      sALS  Excitatory_neurons  cell_class_major_harmony     1770
GSM5292145      sALS  Excitatory_neurons  cell_class_major_harmony      330
GSM5292146     c9ALS  Excitatory_neurons  cell_class_major_harmony     2322
GSM5292147     c9ALS  Excitatory_neurons  cell_class_major_harmony     3094
GSM5292148      sALS  Excitatory_neurons  cell_class_major_harmony     1197
GSM5292156     c9ALS  Excitatory_neurons  cell_class_major_harmony     1632
GSM5292157     c9ALS  Excitatory_neurons  cell_class_major_harmony     2609
GSM5292174   Control  Excitatory_neurons  cell_class_major_harmony     1269
GSM5292175   Control  Excitatory_neurons  cell_class_major_harmony     1278
GSM5292176   Control  Excitatory_ne

## Excitatory-Neuron Focused Refinement

In [ ]:
adata_exc = adata_hm[adata_hm.obs["cell_class_major_harmony"] == "Excitatory_neurons"].copy()
print(adata_exc.shape)
print(adata_exc.obs["leiden_harmony"].value_counts().sort_index())

In [ ]:
umn_marker_dict = {
    "VAT1L_UMN_like": ["VAT1L", "EYA4", "THSD4", "POU3F1", "NEFH", "SCN4B"],
    "SCN4B_projection_like": ["SCN4B", "NEFH", "SYT2", "SV2C"],
    "Broad_L5_exc": ["TLE4", "BCL11B", "FEZF2", "CRYM", "PCP4"],
    "Broad_IT_exc": ["CUX2", "RORB", "THEMIS", "SATB2"],
}

umn_marker_dict = {
    k: [g for g in v if g in adata_exc.var_names]
    for k, v in umn_marker_dict.items()
}
umn_marker_dict = {k: v for k, v in umn_marker_dict.items() if len(v) > 0}

umn_marker_dict

In [ ]:
sc.pl.dotplot(
    adata_exc,
    umn_marker_dict,
    groupby="leiden_harmony",
    standard_scale="var"
)

In [ ]:
umn_marker_panel = [g for g in [
    "VAT1L", "EYA4", "THSD4",
    "SCN4B", "NEFH",
    "TLE4", "BCL11B", "FEZF2", "CRYM", "PCP4",
    "RORB", "THEMIS", "CUX2", "SATB2"
] if g in adata_exc.var_names]

sc.pl.umap(adata_exc, color=umn_marker_panel, ncols=4)

In [ ]:
vat1l_program = [g for g in ["VAT1L", "EYA4", "THSD4", "POU3F1", "NEFH"] if g in adata_exc.var_names]
scn4b_program = [g for g in ["SCN4B", "NEFH", "SYT2", "SV2C"] if g in adata_exc.var_names]
deep_l5_program = [g for g in ["TLE4", "BCL11B", "FEZF2", "CRYM", "PCP4"] if g in adata_exc.var_names]

sc.tl.score_genes(adata_exc, gene_list=vat1l_program, score_name="VAT1L_program_score")
sc.tl.score_genes(adata_exc, gene_list=scn4b_program, score_name="SCN4B_program_score")

sc.pl.umap(
    adata_exc,
    color=["VAT1L_program_score", "SCN4B_program_score", "leiden_harmony"],
    ncols=2
)

In [ ]:
adata_exc.obs["exc_subclass_focus"] = "Other_excitatory"

# best UMN-like candidate
adata_exc.obs.loc[
    adata_exc.obs["leiden_harmony"].astype(str).isin(["22"]),
    "exc_subclass_focus"
] = "UMN_like_VAT1L"

# projection-like candidates
adata_exc.obs.loc[
    adata_exc.obs["leiden_harmony"].astype(str).isin(["11", "18"]),
    "exc_subclass_focus"
] = "Projection_like_SCN4B"

# broad IT-like excitatory
adata_exc.obs.loc[
    adata_exc.obs["leiden_harmony"].astype(str).isin(["12"]),
    "exc_subclass_focus"
] = "Broad_IT_exc"

adata_exc.obs["exc_subclass_focus"] = adata_exc.obs["exc_subclass_focus"].astype("category")

In [ ]:
sc.pl.umap(
    adata_exc,
    color=["exc_subclass_focus", "sample", "condition"],
    wspace=0.35
)

In [1]:
if "merge_key" not in adata_exc.obs.columns:
    adata_exc.obs["barcode"] = adata_exc.obs_names.astype(str)
    adata_exc.obs["merge_key"] = adata_exc.obs["sample"].astype(str) + "__" + adata_exc.obs["barcode"].astype(str)

exc_focus_map = adata_exc.obs.set_index("merge_key")["exc_subclass_focus"].to_dict()

adata_merged_pb.obs["exc_subclass_focus"] = adata_merged_pb.obs["merge_key"].map(exc_focus_map)

print(adata_merged_pb.obs["exc_subclass_focus"].value_counts(dropna=False))

NameError: name 'adata_exc' is not defined

In [ ]:
pb_exc_ckpt = PROJECT_ROOT / "data/processed/merged/adata_merged_pb_exc_focus.h5ad"
pb_exc_ckpt.parent.mkdir(parents=True, exist_ok=True)

adata_merged_pb.write_h5ad(pb_exc_ckpt)

print(f"Saved excitatory-focus-labelled pseudobulk object to: {pb_exc_ckpt}")
print(adata_merged_pb.shape)
print("Has exc_subclass_focus:", "exc_subclass_focus" in adata_merged_pb.obs.columns)